# Pipeline completo: recomendador de libros con IHDT

5 etapas: adquisición, preprocesamiento, división y grafo, entrenamiento y evaluación.


## 0. Preparación

In [ ]:
import os, subprocess, sys

REPO_URL = "https://github.com/FreddyForero1988/Proyecto-Final-Ap_Deep_Learning.git"


# 2. Clonar y entrar a la carpeta del proyecto (si no estás ya dentro)
if not os.path.exists("src"):
    repo_dir = REPO_URL.rstrip("/").split("/")[-1]
    if repo_dir.endswith(".git"):
        repo_dir = repo_dir[:-4]
    if not os.path.exists(repo_dir):
        subprocess.run(["git", "clone", REPO_URL], check=True)
    os.chdir(repo_dir)
    # Si subiste la carpeta completa a GitHub, el proyecto queda en una subcarpeta
    if not os.path.exists("src"):
        subs = [d for d in os.listdir(".") if os.path.exists(os.path.join(d, "src"))]
        if not subs:
            raise FileNotFoundError("No encuentro la carpeta 'src' en el repositorio. "
                                    "Revisa qué archivos subiste a GitHub.")
        os.chdir(subs[0])

print("Carpeta de trabajo:", os.getcwd())

In [ ]:
import torch
print("GPU disponible:", torch.cuda.is_available())

## 1. Parámetros

Los valores por defecto están en `src/config.py`

In [ ]:
from src.config import Config

cfg = Config()
# Prueba rápida
cfg.user_frac, cfg.epochs = 0.05, 2
cfg

## 2. Etapas 1 a 3: adquisición, preprocesamiento, división y grafo

In [ ]:
from src.data import prepare_data

data = prepare_data(cfg)
print(f"Lectores: {data.n_users} | libros: {data.n_items}")
print(f"train/val/test: {len(data.train)}/{len(data.val)}/{len(data.test)}")
print(f"géneros: {len(data.genre_vocab)} | autores: {len(data.author_vocab)}")
data.items.head()

## 3. Etapas 4 y 5: entrenamiento y evaluación

Entrena y evalúa popularidad, MF-BPR, LightGCN e IHDT.

In [ ]:
from src.train import run

results, table = run(["popularity", "mf", "lightgcn", "ihdt"], cfg)
table.round(4)

## 4. Curvas de validación

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from src.config import RESULTS_DIR

fig, ax = plt.subplots(1, 2, figsize=(10, 3.5))
for name in ["mf", "lightgcn", "ihdt"]:
    h = pd.read_csv(RESULTS_DIR / f"history_{name}.csv")
    ax[0].plot(h["epoch"], h["loss"], label=name)
    ax[1].plot(h["epoch"], h[f"recall@{cfg.topk}"], label=name)
ax[0].set_title("Pérdida BPR (entrenamiento)"); ax[1].set_title(f"Recall@{cfg.topk} (validación)")
for a in ax: a.set_xlabel("época"); a.legend()
plt.tight_layout(); plt.show()